# Module 4.2: Full Immich Stack Migration to `homelab-gpu` — CUDA Machine Learning & NVENC Hardware Transcoding

## Homelab Expansion (Chapter 2) - Step 4: GPU Acceleration & Workload Migration

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Articulate why moving the **entire Immich stack** (`immich-server`, `immich-machine-learning:release-cuda`, `immich-postgres`, and `immich-redis`) onto `homelab-gpu` unlocks both **NVIDIA CUDA** AI indexing *and* **NVIDIA NVENC** hardware video transcoding while dedicating `homelab-pi` strictly to Tier-0 network & monitoring infrastructure.
2. Clean up legacy Raspberry Pi artifacts (`/mnt/homelab_storage/swapfile` and the incompatible `aarch64` Postgres data folder) on your internal **WD Blue SN5000** NVMe drive.
3. Deploy the complete 4-container Immich stack on `homelab-gpu` (`10.0.0.201:2283`) configured with official upstream `hwaccel.ml.yml` (`service: cuda`) and `hwaccel.transcoding.yml` (`service: nvenc`).
4. Execute a clean cross-architecture (`aarch64` → `x86_64`) PostgreSQL database restore from `/mnt/homelab_storage/immich_migration_dump.sql.gz` so every album, user, and uploaded photo from Chapter 1 comes right back online.
5. Verify live **CUDA** Smart Search/Face Detection in `nvidia-smi`/`nvtop`, enable **NVENC** video transcoding in Immich Administration, and repoint your browser/mobile clients to `10.0.0.201:2283`.

## 4.2.1 Conceptual Background: Why Full Immich Consolidation Beats Splitting Immich

In Chapter 1, our 4GB Raspberry Pi 4 ran all four Immich services simultaneously on four ARM Cortex-A72 CPU cores. When adding a GPU node (`homelab-gpu`), you could split Immich across two hosts (leaving `immich-server` + Postgres on the Pi and putting only `immich-machine-learning` on the laptop)—**or** consolidate the entire Immich stack onto `homelab-gpu` with your internal **WD Blue SN5000** NVMe drive.

### 1. Why Consolidating All of Immich onto `homelab-gpu` Wins
1. **Unlocks BOTH CUDA AI and NVENC Video Transcoding:** Immich runs FFmpeg video transcoding inside **`immich-server`** (not inside `immich-machine-learning`). Leaving `immich-server` on the Raspberry Pi leaves 4K HEVC video transcoding stuck on ARM CPU. Moving the full stack onto `homelab-gpu` gives `immich-server` direct access to the laptop's **Turing NVENC/NVDEC silicon** (300+ FPS hardware video transcoding) alongside **CUDA** for CLIP & InsightFace!
2. **True Tier-0 Isolation on `homelab-pi`:** With zero heavy databases or photo uploads touching the Raspberry Pi, `homelab-pi` becomes a rock-solid, ~5W **Tier-0 Network & Observability box** (Pi-hole v6 DNS, Tailscale Subnet Router, Uptime Kuma) that stays untouched even under heavy photo/video batch jobs.
3. **Zero Cross-LAN ML Traffic & Simpler Chapter 3 Backups:** `immich-server` talks directly to `immich-machine-learning` over local Docker bridge networking (no need to expose `:3003` across the LAN), and in Chapter 3 every stateful database (`Immich`, `Paperless-ngx`, `Mail-Archiver`, `UniFi Network`) backs up from one node over `homelab-gpu`'s fast link to `UNAS 2`.

### 2. Quantitative Comparison: Raspberry Pi 4 CPU vs. `homelab-gpu` (Internal NVMe + CUDA + NVENC)

| Workload / Metric | Chapter 1 (`homelab-pi`, 4GB ARM + USB SSD) | Chapter 2 (`homelab-gpu`, x86_64 + Internal M.2 NVMe + NVIDIA) | Engineering Impact |
| :--- | :--- | :--- | :--- |
| **CLIP Smart Search Inference (`1,000` Photos)** | ~35 – 50 minutes (`1` worker thread) | **~45 – 75 seconds** (`CUDAExecutionProvider` on 1,536+ CUDA cores) | **~40× speedup** during bulk album imports and re-indexing. |
| **Facial Detection & Recognition (`buffalo_l`)** | ~1.5 – 2.8 seconds per multi-person photo | **~30 – 65 milliseconds** per photo in GDDR6 VRAM | Instant face clustering as soon as photos upload from your phone. |
| **4K HEVC (H.265) Video Transcoding** | ~0.15× real-time (CPU pegged at 400%, thermal throttling) | **3× – 5× real-time** via dedicated **Turing NVENC** silicon encoder | Zero CPU bottleneck when generating web-playable video previews. |
| **Database & Photo I/O** | USB 3.0 UASP bridge (~350–400 MB/s) | **Native PCIe M.2 Slot 2** (1,700–3,500 MB/s + native SMART) | Instant timeline scrolling and zero USB disconnect risk. |
| **Raspberry Pi RAM Used by Immich** | ~2.4 GB – 3.2 GB RAM (+ 2GB SSD swap) | **0 MB on the Pi** | Leaves **>3.5 GB free RAM** on `homelab-pi` purely for DNS, Tailscale & monitoring! |

### 3. Consolidated Full-Stack Architecture on `homelab-gpu`

```mermaid
flowchart LR
    subgraph Pi ["Node 1: homelab-pi (10.0.0.200) — Tier-0 Network Core"]
        NetCore["Pi-hole v6 DNS (:53/:8080)<br>Tailscale + Uptime Kuma (:3001)"]
    end
    
    subgraph Laptop ["Node 2: homelab-gpu (10.0.0.201) — Full Immich + NVIDIA GPU"]
        Server["immich-server (:2283)<br>+ NVIDIA NVENC Transcoding"]
        ML["immich-machine-learning:release-cuda<br>(Internal Docker Bridge | CUDA)"]
        DB["immich-postgres (x86_64)"]
        Redis["immich-redis"]
        SSD["Internal M.2 Slot 2: WD Blue SN5000<br>(/mnt/homelab_storage)"]
        GPU["NVIDIA GTX/RTX GPU<br>(CUDA Cores + Turing NVENC)"]
    end
    
    Server --> DB
    Server --> Redis
    Server --> ML
    Server --> SSD
    DB --> SSD
    Server --> GPU
    ML --> GPU
    NetCore -. "Health Probe (:2283/api/server/ping)" .-> Server
```

## 4.2.2 Step-by-Step Laboratory Guide

### Task 4.2.1: Prepare `/mnt/homelab_storage` & Download Official Immich + GPU Compose Templates
Connect to **`homelab-gpu`** (`ssh homelab-gpu`). First, clean up the obsolete Raspberry Pi artifacts on `/mnt/homelab_storage` (the Pi's 2 GB `swapfile` and the ARM64 binary `immich_postgres` directory—our portable `immich_migration_dump.sql.gz` and all photos in `immich_library/` stay completely untouched!):
```bash
# 1. Double-check your SQL migration dump and photo library are present BEFORE removing the old ARM DB binaries!
ls -lh /mnt/homelab_storage/immich_migration_dump.sql.gz /mnt/homelab_storage/immich_library
gzip -t /mnt/homelab_storage/immich_migration_dump.sql.gz && echo "Dump verified OK!"

# 2. Remove the old Raspberry Pi swapfile and incompatible ARM64 Postgres binary folder
sudo rm -f /mnt/homelab_storage/swapfile
sudo rm -rf /mnt/homelab_storage/docker_volumes/immich_postgres
mkdir -p /mnt/homelab_storage/docker_volumes/immich_postgres
```

Now create the Immich stack directory on `homelab-gpu` and download the official upstream `docker-compose.yml` along with both NVIDIA GPU acceleration templates (`hwaccel.ml.yml` and `hwaccel.transcoding.yml`):
```bash
mkdir -p ~/homelab/stacks/immich
cd ~/homelab/stacks/immich

# 3. Download official Immich Compose + NVIDIA hardware acceleration templates
wget -O docker-compose.yml https://github.com/immich-app/immich/releases/latest/download/docker-compose.yml
wget -O hwaccel.ml.yml https://github.com/immich-app/immich/releases/latest/download/hwaccel.ml.yml
wget -O hwaccel.transcoding.yml https://github.com/immich-app/immich/releases/latest/download/hwaccel.transcoding.yml

# 4. Restore your Chapter 1 .env (or download fresh example.env if starting clean)
cp /mnt/homelab_storage/immich_migration_env.bak ~/homelab/stacks/immich/.env
chmod 600 ~/homelab/stacks/immich/.env
```

Inspect `~/homelab/stacks/immich/.env` (`nano .env`) and verify these settings:
- `UPLOAD_LOCATION=/mnt/homelab_storage/immich_library`
- `DB_DATA_LOCATION=/mnt/homelab_storage/docker_volumes/immich_postgres`
- `IMMICH_VERSION=release`
- `DB_PASSWORD=` *(keep the exact password from Chapter 1!)*
- **Remove or comment out** `MACHINE_LEARNING_WORKERS=1` (that Pi 4GB throttle is no longer needed on your 16GB–32GB GPU server!).

### Task 4.2.2: Enable NVIDIA CUDA (`hwaccel.ml.yml`) & NVENC (`hwaccel.transcoding.yml`) in `docker-compose.yml`
Edit `~/homelab/stacks/immich/docker-compose.yml` (`nano ~/homelab/stacks/immich/docker-compose.yml`) to enable **both** GPU extensions:

1. Under **`immich-server:`**, uncomment the `extends` block and set `service: nvenc`:
   ```yaml
   services:
     immich-server:
       container_name: immich_server
       image: ghcr.io/immich-app/immich-server:${IMMICH_VERSION:-release}
       extends:
         file: hwaccel.transcoding.yml
         service: nvenc
   ```
2. Under **`immich-machine-learning:`**, append **`-cuda`** to the image tag and uncomment `extends` with `service: cuda`:
   ```yaml
     immich-machine-learning:
       container_name: immich_machine_learning
       # Append -cuda for NVIDIA GPU acceleration:
       image: ghcr.io/immich-app/immich-machine-learning:${IMMICH_VERSION:-release}-cuda
       extends:
         file: hwaccel.ml.yml
         service: cuda
   ```
3. Save and validate the merged Docker Compose YAML syntax:
   ```bash
   cd ~/homelab/stacks/immich
   docker compose config --quiet && echo "docker-compose.yml syntax valid!"
   ```

### Task 4.2.3: Restore Your Chapter 1 PostgreSQL Database (`aarch64` → `x86_64`) & Boot Immich on `homelab-gpu`
Follow the official Immich CLI restore sequence so `immich_postgres` initializes clean `x86_64` database files and imports every account, album, and asset record from your Chapter 1 backup:

```bash
cd ~/homelab/stacks/immich

# 1. Pull all x86_64 + CUDA images and create the containers without starting the app server yet
docker compose pull
docker compose create

# 2. Boot ONLY PostgreSQL and wait ~10 seconds for initial database setup
docker start immich_postgres
sleep 10

# 3. Restore your Chapter 1 logical dump (with Immich's official search_path rewrite for vector extensions!)
gunzip --stdout /mnt/homelab_storage/immich_migration_dump.sql.gz \
  | sed "s/SELECT pg_catalog.set_config('search_path', '', false);/SELECT pg_catalog.set_config('search_path', 'public, pg_catalog', true);/g" \
  | docker exec -i immich_postgres psql --dbname=postgres --username=postgres

# 4. Bring up the entire GPU-accelerated Immich stack!
docker compose up -d
docker compose ps
```
Wait ~15 seconds and verify Immich's API ping responds with `pong` on `homelab-gpu` (`10.0.0.201:2283`):
```bash
curl -s http://10.0.0.201:2283/api/server/ping && echo ""
```
*(Expected output: `{"res":"pong"}`)*

### Task 4.2.4: Enable NVENC Hardware Transcoding, Upgrade ML Models & Verify Live CUDA Execution
1. Open your migrated Immich Web UI in a browser at its **new Node 2 address**:
   ```text
   http://10.0.0.201:2283
   ```
   *(or `http://homelab-gpu:2283` over Tailscale!)* Log in with your existing Chapter 1 credentials and confirm all of your previously uploaded photos and albums appear intact.
2. **Enable Hardware NVENC Video Transcoding:**
   - Navigate to **Administration** (top-right gear icon) $\rightarrow$ **Settings** $\rightarrow$ **Video Transcoding Settings** $\rightarrow$ **Hardware Acceleration**.
   - Select **`NVENC`** (and enable **Hardware Decoding**), then click **Save**.
3. **Optional Model Accuracy Upgrade (Powered by 6GB–8GB VRAM):**
   - Under **Administration $\rightarrow$ Settings $\rightarrow$ Machine Learning Settings**, verify **URL** is set to the internal Docker address `http://immich-machine-learning:3003` (zero LAN hops!).
   - Under **Smart Search**, you can upgrade to **`ViT-B-16-SigLIP__webli`** for much richer natural-language search, then click **Save**.
4. **Watch Live CUDA & NVENC Execution:**
   On `homelab-gpu`, tail the ML logs and watch `nvidia-smi` (or `nvtop`) while triggering **Smart Search** / **Face Detection** / **Transcode Videos** under **Administration $\rightarrow$ Jobs**:
   ```bash
   docker logs -f immich_machine_learning
   # In another terminal:
   watch -n 1 nvidia-smi
   ```
   You will see ONNX Runtime initialize **`CUDAExecutionProvider`** in GDDR6 VRAM and FFmpeg invoke **`NVENC`** hardware silicon!
5. **Update Your Smartphone Immich App:** Open the Immich iOS/Android app $\rightarrow$ Settings $\rightarrow$ Server Endpoint URL and update it from `http://homelab-pi:2283/api` (or `10.0.0.200:2283`) to **`http://homelab-gpu:2283/api`** (over Tailscale) or **`http://10.0.0.201:2283/api`** (on LAN).
6. *(Optional cleanup after verifying everything works for a few days:)* Remove `/mnt/homelab_storage/immich_migration_dump.sql.gz` and `/mnt/homelab_storage/immich_migration_env.bak` once Chapter 3 automated backups are live.

### 🖼️ Task 4.2.4 Verification: Migrated Immich on `homelab-gpu:2283` & Live NVIDIA CUDA/NVENC Execution

**Verification Objective:** Capture a screenshot showing `docker compose ps` + `nvidia-smi` (or `docker logs immich_machine_learning` using `CUDAExecutionProvider`) on `homelab-gpu` alongside your Immich Web UI accessible at `http://10.0.0.201:2283` with your Chapter 1 photo library restored.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Legacy Pi `swapfile` and incompatible ARM64 `immich_postgres` directory cleaned from `/mnt/homelab_storage` on `homelab-gpu`.
- [ ] Full 4-container Immich stack deployed on `homelab-gpu` (`10.0.0.201:2283`) with both `hwaccel.transcoding.yml` (`nvenc`) and `hwaccel.ml.yml` (`cuda`) active.
- [ ] Chapter 1 PostgreSQL dump (`immich_migration_dump.sql.gz`) cleanly restored into `x86_64` `immich_postgres` and existing photo library verified.
- [ ] Live NVIDIA CUDA ML execution and NVENC hardware transcoding verified, and mobile app endpoint updated to `homelab-gpu:2283`.
- [ ] Verification screenshot captured and attached above.

Proceed to [Module 5.1: Tier-0 Raspberry Pi Workload Strategy & Pi-hole v6 DNS Deployment](../05_pi_core_services/05_1_pi_workload_strategy_and_pihole.ipynb).